# BatDetect2 UI on Google Colab

這份 notebook 會從 GitHub 取得 BatDetect2、安裝 Python 套件，並在 Colab 中啟動 `batdetect2_ui` 訓練介面。

如要訓練模型，請先選擇 **Runtime → Change runtime type → T4 GPU**。GPU 是否可用仍取決於 Colab 當下提供的 runtime。

## 1. 取得 GitHub repo

預設使用 `pantong552/batdetect2` 的 `main` branch。如果 repo 已經存在於目前 runtime，會嘗試 fast-forward 更新。

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/batdetect2")
if (repo_dir / ".git").exists():
    subprocess.run(
        ["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "https://github.com/pantong552/batdetect2.git", str(repo_dir)],
        check=True,
    )

%cd /content/batdetect2

## 2. 安裝 BatDetect2

安裝 repo 本身及 UI server 需要的 `psutil`。首次安裝可能需要幾分鐘。

In [ ]:
%pip install -e .
%pip install psutil

## 3. 確認 Colab runtime

若沒有 GPU，仍可啟動介面和執行部分工作；訓練通常會慢很多。

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("目前使用 CPU。若需要 GPU，請到 Runtime 設定中選擇 GPU，然後重新執行 notebook。")

## 4. 將 Google Drive 的資料集接到訓練設定

資料夾沿用你現有的單一 dataset 結構，不需要分 `train` 和 `val`。將本機使用的 `dataset.yaml`、`targets.yaml`，以及 `audio/`、`anns/` 放在 Drive 同一個資料夾內：

```text
MyDrive/batdetect2_dataset/
  audio/
  anns/
  dataset.yaml
  targets.yaml
```

掛載 Drive 後，Cell 會讀取 Drive 上的 `dataset.yaml`，將其中音訊目錄設定為 `audio/`、annotation 目錄設定為 `anns/`，並在 Colab repo 根目錄產生可直接使用的 `drive_dataset.yaml`。

In [ ]:
from google.colab import drive
from pathlib import Path
import yaml

drive.mount("/content/drive")

# 修改成 Google Drive 中 batdetect2_dataset 的實際位置。
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/batdetect2_dataset")

dataset_yaml_on_drive = DRIVE_DATA_ROOT / "dataset.yaml"
targets_yaml_on_drive = DRIVE_DATA_ROOT / "targets.yaml"
audio_dir = DRIVE_DATA_ROOT / "audio"
annotations_dir = DRIVE_DATA_ROOT / "anns"

required_paths = [dataset_yaml_on_drive, targets_yaml_on_drive, audio_dir, annotations_dir]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Google Drive 中找不到以下檔案或資料夾：\n" + "\n".join(missing))

dataset_config = yaml.safe_load(dataset_yaml_on_drive.read_text(encoding="utf-8"))
if not isinstance(dataset_config, dict) or not isinstance(dataset_config.get("sources"), list):
    raise ValueError("dataset.yaml 必須包含 sources 清單。")

for source in dataset_config["sources"]:
    if not isinstance(source, dict):
        raise ValueError("dataset.yaml 的每個 source 必須是 mapping。")
    source["audio_dir"] = str(audio_dir)
    if "annotations_dir" in source:
        source["annotations_dir"] = str(annotations_dir)
    elif "annotations_path" in source:
        source["annotations_path"] = str(annotations_dir / Path(source["annotations_path"]).name)
        if not Path(source["annotations_path"]).is_file():
            raise FileNotFoundError(f"找不到 anns/ 中的 annotation 檔案：{source['annotations_path']}")
    else:
        raise ValueError("dataset.yaml source 需包含 annotations_dir 或 annotations_path。")

dataset_path = Path("/content/batdetect2/drive_dataset.yaml")
dataset_path.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")
print(f"Train dataset YAML: {dataset_path}")
print(f"Targets YAML: {targets_yaml_on_drive}")
print("已使用單一 dataset；不會建立 train/val 子資料夾。")

啟動 UI 後，設定以下欄位，再按 Start Training：

- **Train Dataset (YAML)**：`/content/batdetect2/drive_dataset.yaml`
- **Validation Dataset (YAML)**：留空。你的資料集沒有分出獨立 validation set；請勿直接把同一份資料同時當 training 和 validation，否則 validation 指標不能代表未見資料的表現。
- **Target Categories (YAML)**：`/content/drive/MyDrive/batdetect2_dataset/targets.yaml`

這份設定假設 `dataset.yaml` 使用 `batdetect2` 格式（每個錄音一個 JSON，JSON 放在 `anns/`，通常以 `recording.wav.json` 對應 `audio/recording.wav`）。Notebook 也支援 `annotations_path` 指向 `anns/` 中的單一檔案。

## 5. 啟動 BatDetect2 UI

這個 cell 會在背景啟動服務，並檢查 UI API 是否正常回應。若啟動失敗，會顯示 server log 方便排查。

In [ ]:
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

log_path = Path("/tmp/batdetect2_ui.log")
ui_process = globals().get("ui_process")

if ui_process is None or ui_process.poll() is not None:
    ui_log = log_path.open("w")
    ui_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "batdetect2_ui",
            "--host",
            "0.0.0.0",
            "--port",
            "8000",
            "--no-browser",
        ],
        cwd="/content/batdetect2",
        stdout=ui_log,
        stderr=subprocess.STDOUT,
    )

health_url = "http://127.0.0.1:8000/api/system-info"
for attempt in range(30):
    if ui_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            if response.status == 200:
                print("BatDetect2 UI 已啟動。請執行下一格以顯示介面。")
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    print("UI 啟動逾時。以下是 server log：")
    print(log_path.read_text(errors="replace") if log_path.exists() else "找不到 log 檔。")
    raise RuntimeError("BatDetect2 UI 未能在時限內啟動。")

if ui_process.poll() is not None:
    print(log_path.read_text(errors="replace") if log_path.exists() else "找不到 log 檔。")
    raise RuntimeError("BatDetect2 UI 程序已停止。")

## 6. 開啟介面

執行以下 cell，UI 會嵌入在 notebook 中。若嵌入畫面沒有顯示，也可按輸出中的連結在新分頁開啟。

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8000, height=900)
output.serve_kernel_port_as_window(8000)

## 使用與保存結果

- 在介面中上傳錄音和 annotation，並依介面提供的選項設定及執行訓練。
- 輸出預設放在 `/content/batdetect2/outputs`，模型 checkpoint 通常在 `outputs/checkpoints`。
- Colab runtime 會逾時或重設；請在結束前將重要模型及結果複製到 Google Drive 或下載保存。`/content` 不是永久儲存空間。
- 若要重新啟動已停止的 UI，重新執行「啟動 BatDetect2 UI」及「開啟介面」兩格。

In [ ]:
# 可選：將訓練輸出複製到 Google Drive 保存
from google.colab import drive
import shutil

drive.mount("/content/drive")
source = Path("/content/batdetect2/outputs")
destination = Path("/content/drive/MyDrive/batdetect2_outputs")
if source.exists():
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f"已複製輸出至：{destination}")
else:
    print("目前尚無 outputs 目錄。")